In [ ]:
import cipo

# OASI site coordinates
latitude = -8.79225   # 8 deg 47 min 32.1 sec S
longitude = -38.68853  # 38 deg 41 min 18.7 sec W

# Window calculation parameters
start_year = 2025
number_of_years = 5

observation_windows = cipo.calculate_observation_windows(
    latitude, longitude, start_year, number_of_years
)

print("Observation windows (15 days centered on New Moon):\n")
for start_date, new_moon, end_date in observation_windows:
    print(f"Window: {start_date} / {end_date} | New Moon: {new_moon}")


In [ ]:
import pandas as pd


def filter_dynamic_ephemerides(
    ephemerides_by_object,
    start_time,
    end_time,
    altitude_min=10,
    time_min_minutes=30,
):
    '''Filter dynamic ephemerides for valid observation windows.

    Args:
        ephemerides_by_object: Mapping of object names to MPC ephemeris dataframes.
        start_time: Start of the requested period in Brazil time (UTC-3).
        end_time: End of the requested period in Brazil time (UTC-3).
        altitude_min: Minimum object altitude in degrees.
        time_min_minutes: Minimum continuous visibility duration in minutes.

    Returns:
        A mapping of object names to valid ephemeris rows and visibility summaries.
    '''
    filtered_results = {}
    start_time = pd.to_datetime(start_time)
    end_time = pd.to_datetime(end_time)

    for object_name, dataframe in ephemerides_by_object.items():
        processed = dataframe.copy()
        processed['Datetime_UTC'] = pd.to_datetime(
            processed['Date'].astype(str) + ' ' + processed['UT'].astype(str),
            errors='coerce',
        )
        processed = processed.dropna(subset=['Datetime_UTC'])
        processed['Datetime_BRT'] = processed['Datetime_UTC'] - pd.Timedelta(hours=3)

        requested_period = (
            (processed['Datetime_BRT'] >= start_time)
            & (processed['Datetime_BRT'] <= end_time)
        )
        processed = processed.loc[requested_period].copy()
        if processed.empty:
            continue

        processed['Object Alt'] = pd.to_numeric(processed['Object Alt'], errors='coerce')
        processed['Sun Alt'] = pd.to_numeric(processed['Sun Alt'], errors='coerce')
        visible_rows = processed[
            (processed['Object Alt'] >= altitude_min)
            & (processed['Sun Alt'] <= -18)
        ].copy()
        if visible_rows.empty:
            continue

        visible_rows = visible_rows.sort_values('Datetime_BRT')
        time_gap = visible_rows['Datetime_BRT'].diff()
        visible_rows['New_Window'] = (time_gap > pd.Timedelta(hours=2)) | time_gap.isna()
        visible_rows['Window_ID'] = visible_rows['New_Window'].cumsum()

        valid_windows = []
        for _, window_rows in visible_rows.groupby('Window_ID'):
            duration = (
                window_rows['Datetime_BRT'].max() - window_rows['Datetime_BRT'].min()
            ).total_seconds() / 60.0 if len(window_rows) > 1 else 0
            if duration >= time_min_minutes:
                valid_windows.append(window_rows)

        if not valid_windows:
            continue

        final_rows = pd.concat(valid_windows)
        maximum_altitude_index = final_rows['Object Alt'].idxmax()
        maximum_altitude = final_rows.loc[maximum_altitude_index, 'Object Alt']
        maximum_altitude_time = final_rows.loc[maximum_altitude_index, 'Datetime_BRT']
        final_rows = final_rows.drop(
            columns=['Time_Diff', 'New_Window', 'Window_ID', 'Datetime_UTC', 'Datetime_BRT'],
            errors='ignore',
        )
        filtered_results[object_name] = {
            'Ephemerides': final_rows,
            'Max_Alt': maximum_altitude,
            'Max_Alt_Time': maximum_altitude_time.strftime('%Y-%m-%d %H:%M:%S'),
            'Valid_Windows': len(valid_windows),
        }

    return filtered_results
